# 利用 nbformat 生成 CM1 柱坐标动量诊断 Notebook

本 Notebook 用来**自动生成**一个真正做诊断的 Notebook（例如 `cm1_cylindrical_budget_diagnostics.ipynb`）。

生成 Notebook 的内容包括：
- 读取 CM1 输出（风场与九个动量诊断项）
- 从笛卡尔坐标 (x,y) 旋转到柱坐标 (u_r, u_θ)
- 将九个诊断项从 (ub_*, vb_*) 旋转到 (radial, tangential)
- 方位角平均得到 r–z–t 上的 budget
- 对速度做 mean / eddy 分解，并重算切向动量方程中的 mean / eddy 平流项
- 与 CM1 原始预算项对比一致性

> 说明：这样做的好处是，后续你只需要在生成出的诊断 Notebook 里改路径和变量名即可，不用反复手写框架。

## 1. 安装并导入 nbformat

本节说明如何在当前 Python 环境中安装 `nbformat`，并在代码单元中导入。

在 VS Code 终端中运行（如已安装可跳过）：

```bash
pip install nbformat
```

下面的代码单元会导入 `nbformat`，并检查版本。

In [ ]:
# 导入 nbformat（用于创建与写入 .ipynb 文件）
import nbformat
from nbformat.v4 import new_notebook, new_code_cell, new_markdown_cell

print("nbformat version:", nbformat.__version__)

## 2. 准备诊断 Notebook 的代码片段

这一节中，我们用 Python 列表的形式，将“真正做诊断”的 Notebook 的**各个单元内容**整理出来。

为简洁起见：
- 只写出典型框架和关键步骤；
- 文件路径、变量名可以根据你实际的 CM1 输出进行修改（例如 `dataset/cm1out.nc`、`ub_hadv` 等）。

In [ ]:
# 将要生成的“诊断 Notebook”文件名
output_nb_path = "cm1_cylindrical_budget_diagnostics.ipynb"

# 1）诊断 Notebook 的标题与说明（Markdown）
md_intro = """# CM1 柱坐标风场动量预算诊断

本 Notebook 对 CM1 输出的风场与动量预算九项进行如下处理：

1. 读取三维风场与九个诊断项（ub_*, vb_*）。
2. 相对于台风中心从 (x,y) 变换到柱坐标 (r,\theta)。
3. 将九个诊断项从 (ub_*, vb_*) 旋转到径向/切向分量 (P_r, P_\theta)。
4. 对径向/切向预算项做方位角平均，得到 r–z–t 空间的 budget。
5. 对速度做方位角 mean / eddy 分解（u = \bar u + u'），计算涡动通量等量。
6. 重算切向动量方程中的 mean / eddy 平流项，并与 CM1 原始诊断项对比一致性。

> 注意：本 Notebook 假设数据维度类似于 (time, z, y, x)，变量名与路径需要根据你的实际输出做适当修改。
"""

# 2）导入所需库的代码单元
code_imports = """# 导入常用科学计算与绘图库
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# 让图在 Notebook 中内嵌显示
%matplotlib inline

# 如果需要进度条，可选：
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x: x
"""

# 3）读取 CM1 输出与基础信息
code_load = """# === 步骤 1：读取 CM1 输出 ===
# 请根据自己的数据路径与文件名进行修改

# 例：一个包含风场 (u, v, w, p, rho 等) 的三维输出
ds_state = xr.open_dataset('dataset/cm1out.nc')

# 例：一个包含动量九项诊断的输出（若与 state 文件合在一起，可只读一个）
# 包含变量：ub_hadv, ub_vadv, ub_pgrad, ub_cor, ub_hidiff, ub_vidiff, ub_hturb, ub_vturb, ub_rdamp
#           vb_hadv, vb_vadv, vb_pgrad, vb_cor, vb_hidiff, vb_vidiff, vb_hturb, vb_vturb, vb_rdamp
# 若九项就在 ds_state 中，可直接用 ds_state；否则修改为对应文件
try:
    ds_budget = xr.open_dataset('dataset/typhoon_azimuthal_avg_budget.nc')
except FileNotFoundError:
    ds_budget = ds_state

# 根据实际情况检查维度名，例如 ('time','z','y','x') 或 ('t','z','y','x')
print(ds_state)
"""

# 4）确定台风中心与柱坐标 (r, theta)
code_cyl_coords = """# === 步骤 2：确定台风中心并构造柱坐标 (r, theta) ===

# 这里给出一个简单示例：对每个时刻用地面最低压确定中心
# 你也可以用自己的台风轨迹文件替换这一块

# 假定地面气压变量名为 'prs' 或 'p_sfc' 等
prs_var = 'prs' if 'prs' in ds_state else list(ds_state.data_vars)[0]

# 取地面层（如有垂直维度）
if 'z' in ds_state[prs_var].dims:
    prs_sfc = ds_state[prs_var].isel(z=0)
else:
    prs_sfc = ds_state[prs_var]

x = ds_state['x'].values
y = ds_state['y'].values
X, Y = np.meshgrid(x, y, indexing='xy')

centers_x = []
centers_y = []
for it in range(prs_sfc.sizes['time']):
    p2d = prs_sfc.isel(time=it).values
    ij_min = np.unravel_index(np.argmin(p2d), p2d.shape)
    j_min, i_min = ij_min  # 注意顺序 (y, x)
    centers_x.append(X[j_min, i_min])
    centers_y.append(Y[j_min, i_min])

centers_x = np.array(centers_x)
centers_y = np.array(centers_y)

print('中心轨迹样例 (前几个时刻):')
for it in range(min(5, len(centers_x))):
    print(f't={it}: x0={centers_x[it]:.2f}, y0={centers_y[it]:.2f}')

# 这里示意对第一个时刻构造 r, theta（若要对所有时刻精确处理，可在后续循环中更新 x0,y0）
x0 = centers_x[0]
y0 = centers_y[0]

Xc = X - x0
Yc = Y - y0

r = np.sqrt(Xc**2 + Yc**2)
theta = np.arctan2(Yc, Xc)

cos_th = Xc / (r + 1e-12)  # 避免 r=0 处除零
sin_th = Yc / (r + 1e-12)

# 将 r, cos_th, sin_th 添入到 Dataset 方便后续使用
coords_2d = {'y': ds_state['y'], 'x': ds_state['x']}
ds_state = ds_state.assign_coords(r=(coords_2d, r),
                                  cos_th=(coords_2d, cos_th),
                                  sin_th=(coords_2d, sin_th))

ds_budget = ds_budget.assign_coords(r=(coords_2d, r),
                                    cos_th=(coords_2d, cos_th),
                                    sin_th=(coords_2d, sin_th))
"""

# 5）从 (u,v) 旋转到 (u_r, u_theta)
code_rotate_wind = """# === 步骤 3：从 (u, v) 旋转到 (u_r, u_θ) ===

u = ds_state['u']  # 根据实际变量名修改
v = ds_state['v']

cos_th = ds_state['cos_th']
sin_th = ds_state['sin_th']

# xarray 会自动对齐 (time,z,y,x) 与 (y,x) 的坐标
u_r = u * cos_th + v * sin_th
u_theta = -u * sin_th + v * cos_th

u_r.name = 'u_r'
u_theta.name = 'u_theta'

ds_state = ds_state.assign(u_r=u_r, u_theta=u_theta)
print(ds_state[['u_r', 'u_theta']])
"""

# 6）将九项诊断从 (ub_*, vb_*) 旋转到柱坐标
code_rotate_budget = """# === 步骤 4：将动量九项从 (ub_*, vb_*) 旋转到 (radial, tangential) ===

# 定义所有需要旋转的过程名（与 CM1 变量名后缀一致）
processes = [
    'hadv',  # 水平平流
    'vadv',  # 垂直平流
    'pgrad', # 压力梯度
    'cor',   # 科氏
    'hidiff',
    'vidiff',
    'hturb',
    'vturb',
    'rdamp',
]

cos_th = ds_budget['cos_th']
sin_th = ds_budget['sin_th']

for proc in processes:
    ub_name = f'ub_{proc}'
    vb_name = f'vb_{proc}'
    if ub_name not in ds_budget or vb_name not in ds_budget:
        print(f"跳过 {proc}: 在 ds_budget 中未找到 {ub_name} 或 {vb_name}")
        continue

    ub = ds_budget[ub_name]
    vb = ds_budget[vb_name]

    # 旋转到径向与切向
    P_r = ub * cos_th + vb * sin_th
    P_theta = -ub * sin_th + vb * cos_th

    P_r.name = f'{proc}_r'
    P_theta.name = f'{proc}_theta'

    ds_budget = ds_budget.assign({f'{proc}_r': P_r,
                                  f'{proc}_theta': P_theta})

print('已添加的柱坐标诊断变量示例:')
print([v for v in ds_budget.data_vars if v.endswith('_r') or v.endswith('_theta')][:20])
"""

# 7）对径向/切向预算项做方位角平均
code_azimuthal_avg = """# === 步骤 5：对柱坐标预算项做方位角平均 ===

# 这里假设 r 已经是一个 2D 圆柱坐标；为了做方位平均，一般需要把网格按半径分 bin。
# 为简单起见，用 xarray 的 groupby_bins(r, ...) 做示例：

# 先为半径定义一个一维 bin
r_2d = ds_budget['r']
r_max = float(r_2d.max())
nbin = 60
r_bins = np.linspace(0.0, r_max, nbin+1)

# 以切向 budget 为例，对每个过程做 r-bin 平均（相当于近似方位角平均）
proc_theta_vars = [v for v in ds_budget.data_vars if v.endswith('_theta')]

# 这里只示意对第一个时间、所有高度做平均；
# 若要对所有 time 循环，可以再套一层 time 循环或使用 groupby('time').apply
it = 0
sub = ds_budget.isel(time=it)

azim_mean_list = []
for name in proc_theta_vars:
    da = sub[name]
    da_bins = da.groupby_bins(r_2d, r_bins).mean(dim=('y','x'))
    azim_mean_list.append(da_bins)

# 将结果合并到新的 Dataset（仅示意）
ds_theta_azim = xr.merge(azim_mean_list)
print(ds_theta_azim)
"""

# 8）mean / eddy 分解与涡动通量
code_mean_eddy = """# === 步骤 6：mean / eddy 分解，计算涡动通量 ===

# 对 u_r, u_θ, w 做方位角平均（此处仍用 groupby_bins + (y,x) 平均的思路；
# 更精细的实现可以参考你已有的 azimuthal_averaging 脚本。）

u_r = ds_state['u_r']
u_theta = ds_state['u_theta']
w = ds_state['w']  # 根据实际变量名修改

r_2d = ds_state['r']

# 示意：对一个时间片做处理
t_idx = 0
sub_u_r = u_r.isel(time=t_idx)
sub_u_theta = u_theta.isel(time=t_idx)
sub_w = w.isel(time=t_idx)

# 按 r bin 对 (y,x) 平均，得到方位角平均风场
ur_bar = sub_u_r.groupby_bins(r_2d, r_bins).mean(dim=('y','x'))
ut_bar = sub_u_theta.groupby_bins(r_2d, r_bins).mean(dim=('y','x'))
w_bar = sub_w.groupby_bins(r_2d, r_bins).mean(dim=('y','x'))

# 计算扰动：在原网格上 u' = u - \bar u(r)
# 为此，需要将 \bar u(r_bin) 插值/映射回 (y,x) 网格，这里仅示意：

# 先取 r_bin 的中心，便于后续插值
r_bin_centers = 0.5 * (r_bins[:-1] + r_bins[1:])

# TODO: 根据实际需要，对 ur_bar, ut_bar, w_bar 做插值到每个网格点的 r，
# 得到 ur_bar_interp(y,x), ut_bar_interp(y,x), w_bar_interp(y,x)，
# 进而得到 u_r', u_θ', w' 并计算涡动通量 u_r'u_θ', w'u_θ'。

print('ur_bar dims:', ur_bar.dims)
"""

# 9）重算 mean / eddy 平流并与 CM1 诊断对比
code_recompose = """# === 步骤 7：基于 budget.md 中公式重算切向动量方程各项（mean 与 eddy）===

# 这一节需要：
# - 对方位角平均后的 \bar u_r, \bar u_θ, \bar w 在 (r,z) 上做径向/垂直导数；
# - 用公式构造 mean 平流、曲率、科氏项；
# - 用涡动通量 \overline{u_r'u_θ'}, \overline{w'u_θ'} 构造 eddy 通量散度项；
# 再与旋转后的 CM1 hadv/vadv/curv 等诊断做对比。

# 由于实现细节依赖于你具体采用的 (r,z) 网格与差分方案，这里只给出框架注释，
# 你可以根据 budget.md 中写下的解析表达式，将差分公式填入。

# 示例：对 \bar u_θ(r,z) 做径向导数
# d_ut_dr = ut_bar.diff('r_bins') / dr  # 需要先给 r_bins 一个合适的坐标名

# 之后可以用 xarray 的 .differentiate() 或 .diff() 来实现径向/垂直导数。

"""

# 把上述片段整理成列表，后续用 nbformat 写入新的 Notebook
cells_content = [
    ('markdown', md_intro),
    ('code', code_imports),
    ('code', code_load),
    ('code', code_cyl_coords),
    ('code', code_rotate_wind),
    ('code', code_rotate_budget),
    ('code', code_azimuthal_avg),
    ('code', code_mean_eddy),
    ('code', code_recompose),
]

print(f"将要写入 {len(cells_content)} 个单元到 {output_nb_path}")

## 3. 使用 nbformat 创建新的 Notebook 对象

接下来创建一个空的 Notebook 对象，并设置基本元数据（如内核与语言信息）。

In [ ]:
# 创建一个空的 Notebook 对象，并设置基本元数据
nb = new_notebook()

# 可选：设置一些元数据，便于 VS Code / Jupyter 识别
nb.metadata.update({
    "kernelspec": {
        "name": "python3",
        "display_name": "Python 3",
        "language": "python",
    },
    "language_info": {
        "name": "python",
        "pygments_lexer": "ipython3",
    },
})

print("已创建空的 Notebook 对象。")

## 4. 向 Notebook 添加代码与 Markdown 单元

使用前面准备好的 `cells_content` 列表，依次将 Markdown 和代码单元添加到 Notebook 对象中。

In [ ]:
# 根据 cells_content 列表向 Notebook 添加单元
nb.cells = []

for cell_type, content in cells_content:
    if cell_type == 'markdown':
        nb.cells.append(new_markdown_cell(content))
    elif cell_type == 'code':
        nb.cells.append(new_code_cell(content))
    else:
        raise ValueError(f'未知的单元类型: {cell_type}')

print(f"Notebook 目前包含 {len(nb.cells)} 个单元。")

## 5. 将 Notebook 保存为 `.ipynb` 文件

使用 `nbformat.write()` 将构造好的 Notebook 对象写入磁盘上的新 `.ipynb` 文件路径。

In [ ]:
# 将 Notebook 对象写入磁盘
with open(output_nb_path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print(f'已生成诊断 Notebook: {output_nb_path}')